# MP15 — First PyTorch Neural Network

Expanded executable lesson. Prerequisites: preceding mathematics lessons. Derive the calculation, check a hand example, reproduce the controlled experiment and interpret its limits. Learner mastery remains unassessed.

## Setup
Install the repository development dependencies. `python scripts/training_experiments.py` regenerates the experiment using existing checksum-pinned NAIP scenes; the notebook reads its committed record without downloading data. Arrays use sample-by-feature shape and float64.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=Path.cwd()
if not (root/'src').exists(): root=root.parent
sys.path.insert(0,str(root/'src'))
from rssr.training_lab import linear_loss,initialize,mlp_loss,finite_difference_error
record=json.loads((root/'reports/mp13-mp15-training.json').read_text())


## Mathematics and worked example
A tensor stores values and, when requested, participates in an autograd graph. Calling loss.backward() accumulates derivatives into leaf parameter .grad fields. An optimizer changes parameters using those gradients; zero_grad must clear previous gradients before the next independent update. Our comparison uses float64 CPU tensors so it can be checked against the preceding NumPy derivation, with identical initialization, sample order, full-batch loss and learning rate.

For one scalar prediction p=w*x+b at x=2,y=5,w=1,b=0, L=(2-5)²=9, dL/dw=2*(2-5)*2=-12 and dL/db=-6. SGD at .1 gives w=2.2,b=.6. backward computes derivatives; optimizer.step performs the update. They are separate operations.

model.train() and model.eval() change behavior of modules such as dropout and batch normalization. They do not themselves enable or disable gradient recording. torch.no_grad() or inference_mode() controls gradient tracking during inference. A network containing only linear layers and tanh behaves identically in train/eval mode; this lesson explicitly uses dropout to demonstrate a behavioral difference. Dropout with p=.5 zeros samples in training and scales retained values by two; evaluation is identity. This scaling preserves the expectation, not every individual sample.

In [ ]:
import torch
w=torch.tensor(1.,dtype=torch.float64,requires_grad=True); b=torch.tensor(0.,dtype=torch.float64,requires_grad=True)
optimizer=torch.optim.SGD([w,b],lr=.1)
loss=(w*2+b-5)**2; loss.backward()
assert w.grad.item()==-12 and b.grad.item()==-6
optimizer.step()
assert np.isclose(w.item(),2.2) and np.isclose(b.item(),.6)
optimizer.zero_grad(set_to_none=True)
assert w.grad is None
print(w.item(),b.item())

## Executed experiment
The experiment trains the same 1→8→1 tanh network in NumPy and PyTorch for 1000 steps. Its ledger records the maximum final parameter difference, finite-difference error, train/validation loss curves and seed. Agreement validates this small implementation under its declared protocol; it does not imply reproducibility across GPUs, kernels or versions. Dropout is tested separately and is not inserted into the equivalence experiment.

In [ ]:
print(record['optimizer'],record['seed'],record['dtype'])
print('Initial/final toy losses',record['toy_history'][0],record['toy_history'][-1])

## Practice
1. Why does backward twice without zero_grad change .grad? 2. Does eval disable gradients? 3. Why compare identical weights and sample order?

## Worked solutions
1. Gradients accumulate by design. 2. No; use no_grad or inference_mode for inference. 3. Different initialization or stochastic ordering would make numerical differences ambiguous and would not isolate autograd correctness.

In [ ]:
torch.manual_seed(17); d=torch.nn.Dropout(.5); x=torch.ones(64)
d.train(); a=d(x); d.eval(); b=d(x)
assert torch.equal(b,x) and not torch.equal(a,x)
assert record['numpy_torch_final_parameter_max_difference']<1e-10
print(record['numpy_torch_final_parameter_max_difference'],record['dropout'])

## Evidence and interpretation
See [training curves](../reports/mp13-mp15-training.png) and [complete record](../reports/mp13-mp15-training.json). A decreasing training loss is an optimization observation, not proof of geographic generalization. The real red-to-NIR exercise has distinct scene splits and a training-mean baseline, but fixed numerical scaling is not physical calibration. Reflect on a case where the constant baseline could outperform the network.

## References
[PyTorch autograd tutorial](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html) and [optimization tutorial](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html). Models here are compact teaching examples, not paper reproductions or final SR systems.